In [1]:
"""
Simple RAG Pipeline for Support Articles
=========================================
Chunking → Embedding → Retrieval → Grounded Generation

Two retrieval backends:
  1. TF-IDF (zero dependencies beyond scikit-learn, works offline)
  2. OpenAI embeddings (better quality, needs API key)

Swap between them by changing the `retriever` you pass to `generate_answer`.

Requirements:
  pip install scikit-learn anthropic
  pip install openai  # only if using OpenAI embeddings
"""

import json
import hashlib
from dataclasses import dataclass, field
from typing import Optional

# ---------------------------------------------------------------------------
# 1. CHUNKING
# ---------------------------------------------------------------------------

@dataclass
class Chunk:
    text: str
    source: str          # which article it came from
    chunk_index: int
    chunk_id: str = ""

    def __post_init__(self):
        # deterministic ID so we can deduplicate
        raw = f"{self.source}::{self.chunk_index}"
        self.chunk_id = hashlib.md5(raw.encode()).hexdigest()[:12]


def chunk_article(
    text: str,
    source: str,
    max_words: int = 200,
    overlap_words: int = 40,
) -> list[Chunk]:
    """
    Split article text into overlapping word-level chunks.

    Why word-level, not token-level?
      Simpler, portable, and close enough — 200 words ≈ 260 tokens.

    Why overlap?
      A fact that spans a chunk boundary would be lost without it.
      40 words ≈ 2 sentences of shared context.
    """
    words = text.split()
    chunks = []
    start = 0
    index = 0

    while start < len(words):
        end = start + max_words
        chunk_text = " ".join(words[start:end])
        chunks.append(Chunk(text=chunk_text, source=source, chunk_index=index))
        start += max_words - overlap_words   # slide forward with overlap
        index += 1

    return chunks


def build_chunk_store(articles: dict[str, str], **kwargs) -> list[Chunk]:
    """articles = {"article_title": "full text", ...}"""
    all_chunks = []
    for title, body in articles.items():
        all_chunks.extend(chunk_article(body, source=title, **kwargs))
    print(f"Built {len(all_chunks)} chunks from {len(articles)} articles")
    return all_chunks


# ---------------------------------------------------------------------------
# 2. RETRIEVAL — Option A: TF-IDF (no external API needed)
# ---------------------------------------------------------------------------

class TFIDFRetriever:
    """
    Bag-of-words retrieval using sklearn's TF-IDF + cosine similarity.

    Good enough for keyword-heavy support articles.
    Fails on semantic paraphrases ("can't log in" vs "authentication issue").
    """

    def __init__(self, chunks: list[Chunk]):
        from sklearn.feature_extraction.text import TfidfVectorizer

        self.chunks = chunks
        self.vectorizer = TfidfVectorizer(stop_words="english")
        corpus = [c.text for c in chunks]
        self.matrix = self.vectorizer.fit_transform(corpus)

    def retrieve(self, query: str, top_k: int = 3) -> list[Chunk]:
        from sklearn.metrics.pairwise import cosine_similarity

        query_vec = self.vectorizer.transform([query])
        scores = cosine_similarity(query_vec, self.matrix).flatten()

        top_indices = scores.argsort()[::-1][:top_k]
        return [(self.chunks[i], float(scores[i])) for i in top_indices]


# ---------------------------------------------------------------------------
# 2. RETRIEVAL — Option B: OpenAI Embeddings (better semantic matching)
# ---------------------------------------------------------------------------

class EmbeddingRetriever:
    """
    Dense retrieval using OpenAI's embedding model.
    Handles semantic similarity ("can't log in" ≈ "authentication issue").

    Swap in any embedding provider by replacing `_embed`.
    """

    def __init__(self, chunks: list[Chunk]):
        import numpy as np
        self.np = np
        self.chunks = chunks
        texts = [c.text for c in chunks]
        self.embeddings = self.np.array(self._embed_batch(texts))

    def _embed_batch(self, texts: list[str]) -> list[list[float]]:
        from openai import OpenAI
        client = OpenAI()
        resp = client.embeddings.create(
            model="text-embedding-3-small",
            input=texts,
        )
        return [item.embedding for item in resp.data]

    def _embed(self, text: str) -> list[float]:
        return self._embed_batch([text])[0]

    def retrieve(self, query: str, top_k: int = 3) -> list[tuple[Chunk, float]]:
        q_emb = self.np.array(self._embed(query))
        # cosine similarity
        sims = self.embeddings @ q_emb / (
            self.np.linalg.norm(self.embeddings, axis=1) * self.np.linalg.norm(q_emb)
        )
        top_indices = sims.argsort()[::-1][:top_k]
        return [(self.chunks[i], float(sims[i])) for i in top_indices]


# ---------------------------------------------------------------------------
# 3. GENERATION — grounded answer using Anthropic
# ---------------------------------------------------------------------------

def generate_answer(
    question: str,
    retriever,  # anything with a .retrieve(query, top_k) method
    top_k: int = 3,
    model: str = "claude-sonnet-4-6",
) -> dict:
    """
    Retrieve top-k chunks, feed them as context, generate a grounded answer.
    Returns the answer, the chunks used, and retrieval scores.
    """
    import anthropic

    results = retriever.retrieve(question, top_k=top_k)

    # Build context block for the prompt
    context_parts = []
    for i, (chunk, score) in enumerate(results, 1):
        context_parts.append(
            f"[Source {i}: {chunk.source} (chunk {chunk.chunk_index})]"
            f"\n{chunk.text}"
        )
    context = "\n\n---\n\n".join(context_parts)

    prompt = f"""You are a support agent. Answer the customer's question using ONLY
the context below. If the context doesn't contain the answer, say so — do not guess.

When you use information from a source, cite it as [Source N].

Context:
{context}

Customer question: {question}

Answer:"""

    client = anthropic.Anthropic()
    response = client.messages.create(
        model=model,
        max_tokens=500,
        messages=[{"role": "user", "content": prompt}],
    )

    return {
        "question": question,
        "answer": response.content[0].text,
        "sources": [
            {
                "text": chunk.text[:200] + "...",
                "source": chunk.source,
                "score": round(score, 4),
            }
            for chunk, score in results
        ],
    }


# ---------------------------------------------------------------------------
# 4. DEMO — sample knowledge base and end-to-end run
# ---------------------------------------------------------------------------

SAMPLE_ARTICLES = {
    "Password Reset": """
        If you forgot your password, go to the login page and click "Forgot Password".
        Enter the email address associated with your account. You will receive a reset
        link within 5 minutes. The link expires after 24 hours. If you don't receive
        the email, check your spam folder. If the problem persists, contact support
        and we can manually reset your password after verifying your identity with
        your account security questions. Note that passwords must be at least 12
        characters and contain one uppercase letter, one number, and one special
        character. You cannot reuse any of your last 5 passwords.
    """,
    "Billing and Refunds": """
        We bill on the 1st of each month. You can view your invoices in the Billing
        section of your account dashboard. If you believe you were charged incorrectly,
        you can dispute the charge within 30 days by opening a billing ticket. Refunds
        are processed within 5-7 business days and will appear on the original payment
        method. For annual plans, we offer prorated refunds for unused months if you
        cancel before the renewal date. We accept Visa, Mastercard, and ACH transfers.
        Cryptocurrency payments are not supported at this time. If your payment fails,
        your account enters a 7-day grace period before suspension.
    """,
    "Account Suspension": """
        Accounts may be suspended for several reasons: unpaid invoices past the 7-day
        grace period, violation of our terms of service, or suspicious activity detected
        by our security team. If your account is suspended, you will receive an email
        explaining the reason. For billing-related suspensions, simply update your
        payment method and pay the outstanding balance to reactivate immediately. For
        security-related suspensions, you will need to contact our security team at
        security@example.com and verify your identity. During suspension, your data
        is preserved for 90 days. After 90 days, data may be permanently deleted.
        You can export your data within the first 30 days of suspension by contacting
        support.
    """,
    "API Rate Limits": """
        Our API enforces rate limits to ensure fair usage. Free tier accounts are
        limited to 100 requests per minute and 10,000 requests per day. Pro accounts
        get 1,000 requests per minute and 100,000 per day. Enterprise accounts have
        custom limits. When you exceed the rate limit, the API returns a 429 status
        code with a Retry-After header indicating how many seconds to wait. We
        recommend implementing exponential backoff in your client code. You can monitor
        your current usage via the /api/usage endpoint. If you consistently need higher
        limits, contact sales to discuss an enterprise plan. Burst allowances of 2x
        the per-minute limit are permitted for up to 10 seconds.
    """,
}


def main():
    # Build chunks
    chunks = build_chunk_store(SAMPLE_ARTICLES, max_words=80, overlap_words=20)

    # Pick retriever (TF-IDF works without any API keys)
    retriever = TFIDFRetriever(chunks)
    # retriever = EmbeddingRetriever(chunks)  # uncomment for better quality

    # Ask questions
    questions = [
        "My account got suspended and I don't know why. How do I get back in?",
        "I'm getting a 429 error from your API. What do I do?",
        "Can I get a refund if I cancel my annual plan halfway through?",
    ]

    for q in questions:
        print(f"\n{'='*60}")
        print(f"Q: {q}")
        print(f"{'='*60}")
        result = generate_answer(q, retriever)
        print(f"\nA: {result['answer']}")
        print(f"\nSources used:")
        for s in result["sources"]:
            print(f"  - {s['source']} (score: {s['score']})")


if __name__ == "__main__":
    main()

Built 8 chunks from 4 articles

Q: My account got suspended and I don't know why. How do I get back in?


ModuleNotFoundError: No module named 'anthropic'